In [1]:
import pandas as pd
import numpy as np 
df=pd.read_csv('email.csv')

df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
df['Category'].value_counts()

Category
ham               4825
spam               747
{"mode":"full"       1
Name: count, dtype: int64

In [20]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords

# Stopwords download (once)
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

def full_preprocess_sms(df):
    """
    Complete preprocessing for SMS spam/ham dataset.
    Returns cleaned dataframe with columns:
    - label (0=ham, 1=spam)
    - clean_text (ready for vectorization)
    """
    
    # 1️⃣ Strip whitespace from labels
    df["Category"] = df["Category"].astype(str).str.strip()
    
    # 2️⃣ Remove rows with NaN in text or label
    df = df.dropna(subset=["Message", "Category"])
    
    # 3️⃣ Encode labels (ham=0, spam=1)
    df["label"] = df["Category"].map({"ham": 0, "spam": 1})
    
    # Drop rows where mapping failed (any incorrect label)
    df = df.dropna(subset=["label"])
    
    # 4️⃣ Define text cleaning function
    def clean_text(text):
        text = str(text).lower()  # lowercase
        text = re.sub(r"http\S+|www\S+", "", text)  # remove URLs
        text = re.sub(r'\d+', '', text)  # remove numbers
        text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)  # remove punctuation
        text = re.sub(r"[^a-zA-Z\s]", " ", text)  # remove non-alphabetic chars
        text = re.sub(r"\s+", " ", text).strip()  # remove extra spaces
        # remove stopwords
        text = " ".join([w for w in text.split() if w not in stop_words])
        return text
    
    # 5️⃣ Apply cleaning
    df["clean_text"] = df["Message"].apply(clean_text)
    
    return df[["label", "clean_text"]]


[nltk_data] Downloading package stopwords to C:\Users\LAPTOPS
[nltk_data]     HUB\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [14]:
df.head()

,label,text,clean_text
0,0.0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,0.0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1.0,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,0.0,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,0.0,"Nah I don't think he goes to usf, he lives aro...",nah think goes usf lives around though


In [21]:
# Load your dataset
df = pd.read_csv("email.csv")

# Preprocess completely
processed_df = full_preprocess_sms(df)

# Check top rows
processed_df.head()


,label,clean_text
0,0.0,go jurong point crazy available bugis n great ...
1,0.0,ok lar joking wif u oni
2,1.0,free entry wkly comp win fa cup final tkts st ...
3,0.0,u dun say early hor u c already say
4,0.0,nah think goes usf lives around though


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

# Models
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 1️⃣ Split data
X = processed_df["clean_text"]
y = processed_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2️⃣ Vectorize text
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 3️⃣ Define multiple models
models = {
    "MultinomialNB": MultinomialNB(),
    "BernoulliNB": BernoulliNB(),
    "Logistic Regression": LogisticRegression(max_iter=200),
    "SVM Linear": SVC(kernel="linear"),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier()
}

# 4️⃣ Train & evaluate all models
results = {}

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name} Accuracy: {acc:.4f}")

# 5️⃣ Select the best model
best_model_name = max(results, key=results.get)
best_model_accuracy = results[best_model_name]
print("\nBest Model:", best_model_name, "→ Accuracy:", best_model_accuracy)




MultinomialNB Accuracy: 0.9731
BernoulliNB Accuracy: 0.9749
Logistic Regression Accuracy: 0.9641
SVM Linear Accuracy: 0.9883
Decision Tree Accuracy: 0.9507
Random Forest Accuracy: 0.9785

Best Model: SVM Linear → Accuracy: 0.9883408071748879
